# Linear regression: normal equations vs HHL-style

Solve `Xᵀ X β = Xᵀ y` for least-squares regression. Same `apply_linear(M, β)` trace, two routes.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Problem:** least-squares fit of `y ≈ X β`. **Classical:** normal equations or `numpy.linalg.lstsq`. **Quantum:** HHL-style on the SPD normal-equations matrix. The cpu route runs the normal-equations solve; cpusim engages qsim on the apply_linear block.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.ops import linalg, shape
from uniqx.ops.primitives.evolution import apply_linear

rng = np.random.default_rng(3)
n_samples, n_features = 6, 3
X = rng.standard_normal((n_samples, n_features))
init_vec = np.ones(n_features)

@trace
def prog(features, vec):
    Xt = shape.transpose(features, permutation=(1, 0))
    M = linalg.matmul(Xt, features)
    return apply_linear(M, vec)

module = prog(X.tolist(), init_vec.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
